# Kelman Filtering

In [ ]:
!pip install websocket-client

In [2]:
import os
import sys
import json
import numpy as np

current_dir = os.path.dirname(os.path.abspath('KelmanFiltering.ipynb'))
libs_dir = os.path.join(current_dir, '..', 'libs')
libs_dir = os.path.abspath(libs_dir)
print("Libs directory:", os.path.abspath(libs_dir))
sys.path.append(libs_dir)

from websocketclient import WebSocketCollector

Libs directory: /Users/kolithawarnakulasooriya/Projects/AI-driven-cyber-physical-system/libs


In [6]:
ws_url = "ws://localhost:8000/ws"
message_queue = WebSocketCollector(ws_url, max_messages=50).collect()
print("Collected messages:", message_queue.qsize())

WebSocket opened
WebSocket closed
Collected messages: 50


In [7]:
print("Sample message:", message_queue.get())

raw_values = [message_queue.get() for _ in range(message_queue.qsize())]
values = [json.loads(value).get("data").get("value") for value in raw_values]
values = [float(value) for value in values if value is not None]
print("Extracted values:", values)

Sample message: {"type": "initial_state", "data": {"sensors": [{"id": "f868cfdf-7597-42e6-b592-06bac9b4b17e", "name": "Temperature S1", "type": "gaussian", "interval": 0.1, "min_value": 0, "max_value": 100, "is_running": false, "current_value": 77.31998405591891, "recording": false, "parameters": {}}, {"id": "e67764a0-3740-4ffa-8372-4ec634821407", "name": "s1", "type": "lidar", "interval": 1, "min_value": 0, "max_value": 100, "is_running": true, "current_value": 45.68265414440979, "recording": false, "parameters": {}}], "mqtt_connected": false, "recordings": {}}}
Extracted values: [45.3672577706837, 56.65895761414126, 58.070927155816946, 57.13106372089894, 51.61708736570374, 52.25630171547328, 54.030677689347215, 51.1134288281002, 45.04814311540137, 42.269966962627024, 45.103981696809115, 43.29999067363357, 40.9129716025524, 41.29579497082743, 44.26505449818522, 38.40546535169496, 38.12271560141984, 44.04850394304035, 57.62728444599268, 64.79303821722056, 55.20342279151872, 51.63545906

In [8]:
class KalmanFilter1D:
    """
    1D Kalman Filter with state-space representation.
    
    State vector: x = [position, velocity]^T
    
    Process model:
        x_{k|k-1} = F * x_{k-1|k-1} + w_k, where w ~ N(0, Q)
    
    Measurement model:
        z_k = H * x_{k|k-1} + v_k, where v ~ N(0, R)
    
    Parameters:
    -----------
    dt : float
        Time step between measurements
    process_variance : float
        Process noise covariance (Q)
    measurement_variance : float
        Measurement noise covariance (R)
    initial_position : float
        Initial position estimate
    initial_velocity : float
        Initial velocity estimate
    initial_error_covariance : float
        Initial state error covariance P_0
    """
    
    def __init__(self, dt=1.0, process_variance=1e-5, measurement_variance=1e-2, 
                 initial_position=None, initial_velocity=0.0, initial_error_covariance=1.0):
        self.dt = dt
        self.q = process_variance  # Process noise variance
        self.r = measurement_variance  # Measurement noise variance
        
        # State transition matrix (constant velocity model)
        self.F = np.array([[1.0, dt],
                          [0.0, 1.0]])
        
        # Measurement matrix (we only measure position)
        self.H = np.array([[1.0, 0.0]])
        
        # Process noise covariance matrix
        self.Q = np.array([[self.q, 0.0],
                          [0.0, self.q]])
        
        # Measurement noise covariance
        self.R = np.array([[self.r]])
        
        # Initial state: [position, velocity]
        self.x = np.array([[initial_position if initial_position is not None else 0.0],
                          [initial_velocity]])
        
        # Initial state error covariance
        self.P = np.eye(2) * initial_error_covariance
        
        self.estimates = []
        self.velocities = []
        
    def predict(self):
        """Prediction step: x_{k|k-1} = F * x_{k-1|k-1}"""
        self.x = self.F @ self.x
        self.P = self.F @ self.P @ self.F.T + self.Q
        
    def update(self, z):
        """Update step with measurement z"""
        # Innovation (measurement residual)
        y = z - self.H @ self.x
        
        # Innovation covariance
        S = self.H @ self.P @ self.H.T + self.R
        
        # Kalman gain
        K = self.P @ self.H.T / S
        
        # State update
        self.x = self.x + K @ y
        
        # Covariance update
        self.P = (np.eye(2) - K @ self.H) @ self.P
        
    def filter(self, measurements):
        """Apply Kalman filter to a sequence of measurements"""
        self.estimates = []
        self.velocities = []
        
        for measurement in measurements:
            self.predict()
            self.update(np.array([[measurement]]))
            self.estimates.append(self.x[0, 0])
            self.velocities.append(self.x[1, 0])
            
        return self.estimates, self.velocities
    
    def get_state(self):
        """Return current state [position, velocity]"""
        return self.x.flatten()

# Enhanced Kalman filter using class
if 'values' in locals() and values:
    # Use advanced Kalman filter with velocity estimation
    kf = KalmanFilter1D(dt=1.0, 
                       process_variance=1e-5, 
                       measurement_variance=1e-2,
                       initial_position=values[0],
                       initial_velocity=0.0)
    
    filtered_values, velocities = kf.filter(values)
    
    print("=== Advanced Kalman Filter Results ===")
    print(f"Raw values (first 10): {values[:10]}")
    print(f"Filtered values (first 10): {[f'{v:.4f}' for v in filtered_values[:10]]}")
    print(f"Estimated velocities (first 10): {[f'{v:.4f}' for v in velocities[:10]]}")
    print(f"Mean squared error (Filter vs Raw): {np.mean([(f - r)**2 for f, r in zip(filtered_values, values)]):.6f}")


=== Advanced Kalman Filter Results ===
Raw values (first 10): [45.3672577706837, 56.65895761414126, 58.070927155816946, 57.13106372089894, 51.61708736570374, 52.25630171547328, 54.030677689347215, 51.1134288281002, 45.04814311540137, 42.269966962627024]
Filtered values (first 10): ['45.3673', '56.4469', '59.6551', '59.7976', '56.3691', '54.8351', '54.7421', '53.4439', '50.2652', '47.1049']
Estimated velocities (first 10): ['0.0000', '10.7629', '6.2742', '3.6591', '1.3006', '0.5259', '0.3820', '0.0406', '-0.5437', '-0.9757']
Mean squared error (Filter vs Raw): 51.294139
